# Customer Clustering, Replicated on GoiEner

Exploratory notebook, not part of any book chapter. Same purpose as
`04-customer-feeder-clustering-london-code.ipynb`: replicate
`04-customer-feeder-clustering-code.ipynb`'s customer-level clustering
pipeline on a second, independently metered real population, this time
GoiEner (25,559 anonymized Spanish households, Basque Country/Navarre-heavy,
Quesada et al. 2024, *Scientific Data*, Zenodo record 7362094, CC-BY-4.0).

One real, disclosed mismatch this notebook does not paper over: GoiEner is
**hourly**, not half-hourly. Every AusNet/London cell in this pipeline
assumes `STEPS_PER_DAY` steps per day; here that is 24, not 48, so cluster
shapes below are directly comparable to AusNet/London only as *daily
patterns*, not at matching time resolution.

Every AusNet-specific section (DER voltage risk against the real OpenDSS
network, the controlled EV-adoption test, feeder-level SMART-DS
clustering) is skipped here too: no equivalent PV, EV, or feeder-topology
data exists for these GoiEner households.

## Getting the data

`scripts/fetch_goiener_data.py` vendors the real, official `imp-post.tzst`
archive (already-imputed post-2021-05-30 snapshot, 17,519 series) and
`metadata.csv` from Zenodo. `metadata.csv`'s own `missing_samples_pct`,
`length_years`, `start_date`, and `end_date` fields pick real candidates
covering a real 360-day window (2021-06-06 through 2022-05-31, the same
4x90-day quarter convention the AusNet/London notebooks use) before a
single streaming pass through the compressed archive (`zstandard` +
`tarfile`, not a full extraction) pulls out 2,000 real households' own
CSVs, sampled stratified by residential/commercial status (CNAE 9820
marks a household, not a business) so the larger sample keeps the real
population's own ~78/22 mix. A second, per-household real-coverage check
after streaming (not just the metadata-level date range) confirms genuine
hourly completeness, since metadata's own start/end dates do not
guarantee zero gaps inside.

In [ ]:
import io
from pathlib import Path
import tarfile

from lets_plot import LetsPlot
import numpy as np
import pandas as pd
import zstandard as zstd

LetsPlot.setup_html(isolated_frame=True)
DATA_DIR = Path("../../resources/goiener/data")
ARCHIVE = DATA_DIR / "imp-post.tzst"
METADATA = DATA_DIR / "metadata.csv"
STEPS_PER_DAY = 24  # real, disclosed mismatch: GoiEner is hourly, not half-hourly like AusNet/London
N_HOUSEHOLDS = 2000
WINDOW_START = pd.Timestamp("2021-06-06")
WINDOW_DAYS = 360  # 4 real 90-day quarters, the same convention the AusNet/London notebooks use
MIN_COVERAGE = 0.99

if not ARCHIVE.exists():
    raise SystemExit(f"{ARCHIVE} not found; run scripts/fetch_goiener_data.py first")

meta = pd.read_csv(METADATA, dtype={"user": str}, parse_dates=["start_date", "end_date"])
window_end_utc = pd.Timestamp(WINDOW_START, tz="UTC") + pd.Timedelta(days=WINDOW_DAYS)
candidates = meta[
    (meta["missing_samples_pct"] < 1.0)
    & (meta["length_years"] >= 1.0)
    & (meta["start_date"] <= pd.Timestamp(WINDOW_START, tz="UTC"))
    & (meta["end_date"] >= window_end_utc)
]
print(f"real candidates covering the target window at the metadata level: {len(candidates)}")

candidates = candidates.copy()
candidates["is_residential"] = candidates["cnae"] == 9820.0  # CNAE 9820 marks a household, not a business
frac = N_HOUSEHOLDS / len(candidates)
target_ids = set(
    candidates.groupby("is_residential", group_keys=False)[["user"]].apply(
        lambda g: g.sample(frac=frac, random_state=42)
    )["user"]
)  # stratified by residential/commercial so the larger sample keeps the real population's own mix

real candidates covering the target window at the metadata level: 16888


In [ ]:
found = {}
dctx = zstd.ZstdDecompressor()
with ARCHIVE.open("rb") as fh, dctx.stream_reader(fh) as reader, tarfile.open(fileobj=reader, mode="r|") as tar:
    for member in tar:
        if not member.isfile():
            continue
        stem = Path(member.name).stem
        if stem not in target_ids:
            continue
        raw = tar.extractfile(member)
        if raw is None:
            continue
        found[stem] = raw.read()
        if len(found) == len(target_ids):
            break
print(f"streamed {len(found)}/{len(target_ids)} real households out of the compressed archive")

streamed 2000/2000 real households out of the compressed archive


In [ ]:
window_end = WINDOW_START + pd.Timedelta(days=WINDOW_DAYS)
full_index = pd.date_range(WINDOW_START, window_end, freq="1h", inclusive="left")
print(f"expected real hourly timesteps: {len(full_index)}")

series = {}
for uid, raw in found.items():
    df = pd.read_csv(io.BytesIO(raw), parse_dates=["timestamp"]).set_index("timestamp").sort_index()
    win = df.reindex(full_index)
    if win["kWh"].notna().mean() >= MIN_COVERAGE:
        series[uid] = win["kWh"].interpolate(method="linear", limit_direction="both").to_numpy()

load_data = np.stack(list(series.values())).reshape(len(series), WINDOW_DAYS, STEPS_PER_DAY)
n_customers = load_data.shape[0]
print(f"load_data: {load_data.shape} (customers, days, hours), units kWh per hour")

variance = load_data.var(axis=(1, 2))
n_degenerate = int((variance < 1e-6).sum())
print(f"near-zero-variance households (real data-quality check, matching AusNet/London precedent): {n_degenerate}")

expected real hourly timesteps: 8640


load_data: (2000, 360, 24) (customers, days, hours), units kWh per hour
near-zero-variance households (real data-quality check, matching AusNet/London precedent): 16


In [ ]:
from lets_plot import LetsPlot, aes, geom_line, ggplot, ggsize, labs

from ark.plot.theme import modern_theme
from ark.plot.tokens import PRIMARY

LetsPlot.setup_html(isolated_frame=True)

selected_day = np.random.randint(0, load_data.shape[1])  # random day index
day_profiles = load_data[:, selected_day, :]  # (342 customers, 48 half-hours)

profile_long = (
    pd.DataFrame(day_profiles)
    .reset_index(names="customer")
    .melt(id_vars="customer", var_name="half_hour", value_name="kw")
)
profile_long["half_hour"] = profile_long["half_hour"].astype(int)

(
    ggplot(profile_long, aes(x="half_hour", y="kw", group="customer"))
    + geom_line(color="#0369A1", alpha=0.1, size=0.5)
    + labs(x="Half-hour of day", y="Active power (kW)", title=f"All 342 real customers, day {selected_day}")
    + modern_theme()
    + ggsize(650, 320)
)

### K-means as the baseline method

Same pipeline as the AusNet and London notebooks: normalize each real
household's own season-mean daily shape by its own peak.

In [ ]:
from ark.cluster.preprocessing import map_to_seasons, normalize_shape

# Real calendar summer (Northern Hemisphere), not an arbitrary first-90-day
# window: WINDOW_START is 2021-06-06, so the old `load_data[:, 0:90, :]`
# slice actually landed mostly in spring/early summer. Selecting real summer
# days directly keeps this comparable with AusNet's own Southern-Hemisphere
# summer-heavy Q1 window, rather than comparing two different real seasons
# under the same "season" label.
day_index = pd.date_range(WINDOW_START, periods=load_data.shape[1], freq="D")
summer_days = np.where(map_to_seasons(day_index, hemisphere="north") == "summer")[0]
season = load_data[:, summer_days, :]
X = normalize_shape(season.mean(axis=1))
print(f"clustering input: {X.shape} (customers, hourly steps)")

clustering input: (2000, 24) (customers, hourly steps)


In [ ]:
from ark.cluster.cluster_validation import clustering_validity_scores
from ark.plot.clustering import validity_curve
from ark.plot.gt_style import themed_gt

scores = clustering_validity_scores(X, range(2, 10))
themed_gt(scores.round(3))

k,inertia,silhouette,calinski_harabasz,davies_bouldin
2,2055.356,0.223,729.222,1.531
3,1699.185,0.227,650.117,1.473
4,1507.37,0.238,572.983,1.356
5,1362.761,0.172,528.025,1.561
6,1247.365,0.184,498.161,1.476
7,1178.251,0.152,458.75,1.635
8,1129.129,0.148,422.494,1.702
9,1093.99,0.136,389.359,1.782


In [ ]:
from lets_plot import ggsize

p = validity_curve(scores)
p + ggsize(width=600, height=400)

The chosen $k$ below is read directly off the real validity curve above
for this population, not carried over from AusNet's own choice of 5 or
London's own choice of 2.

In [ ]:
N_CLUSTERS = int(scores.loc[scores["silhouette"].idxmax(), "k"])
print(f"N_CLUSTERS chosen from the real silhouette maximum above: {N_CLUSTERS}")

from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=N_CLUSTERS, n_init=20, random_state=0)
labels = kmeans.fit_predict(X)
table = pd.DataFrame({"labels": labels}).value_counts().sort_index().reset_index()
table.columns = ["Label", "Count"]
themed_gt(table)

N_CLUSTERS chosen from the real silhouette maximum above: 4


Label,Count
0,1153
1,66
2,546
3,235


In [ ]:
from ark.plot.clustering import cluster_profiles

p = cluster_profiles(
    X,
    labels,
    x=np.arange(STEPS_PER_DAY) * 1.0,
    x_label="Hour of day",
    y_label="Normalized demand",
    title="Real GoiEner customer archetypes: member profiles and cluster mean",
)
p + ggsize(width=600, height=400)

## How much does that overlap actually matter?

Same conformal-style check as the AusNet and London notebooks: hold back
30% of real households, calibrate a distance-to-centroid threshold at 90%
confidence on them, then see how many archetypes a genuinely honest
confidence set assigns each held-in household.

In [ ]:
from itertools import combinations

from sklearn.model_selection import train_test_split

train_idx, calibration_idx = train_test_split(np.arange(n_customers), test_size=0.3, random_state=0)

conformal_km = KMeans(n_clusters=N_CLUSTERS, n_init=20, random_state=0)
conformal_km.fit(X[train_idx])
centroids = conformal_km.cluster_centers_


def distance_to_centroids(profiles: np.ndarray) -> np.ndarray:
    return np.linalg.norm(profiles[:, None, :] - centroids[None, :, :], axis=2)


ALPHA = 0.1
calibration_scores = distance_to_centroids(X[calibration_idx]).min(axis=1)
n_calibration = len(calibration_idx)
quantile_level = min(np.ceil((n_calibration + 1) * (1 - ALPHA)) / n_calibration, 1.0)
tau = np.quantile(calibration_scores, quantile_level)
print(f"calibrated on {n_calibration} held-back households, threshold tau={tau:.3f}")

calibrated on 600 held-back households, threshold tau=1.276


In [ ]:
all_distances = distance_to_centroids(X)
prediction_sets = [np.where(row <= tau)[0] for row in all_distances]
set_sizes = pd.Series([len(s) for s in prediction_sets])

size_counts = set_sizes.value_counts().sort_index().reset_index()
size_counts.columns = ["Set size", "Households"]
themed_gt(size_counts)

Set size,Households
0,155
1,638
2,1194
3,13


In [ ]:
from lets_plot import aes, geom_bar, ggplot, labs

from ark.plot.theme import BRAND_PALETTE, modern_theme

p = (
    ggplot(size_counts, aes(x="Set size", y="Households"))
    + geom_bar(stat="identity", fill=BRAND_PALETTE[0])
    + labs(
        x="Archetypes in the 90%-confidence set",
        y="Households",
        title="How many archetypes fit each household, at 90% confidence",
    )
    + modern_theme(legend_pos="none")
    + ggsize(600, 400)
)
p

## Does a more advanced method earn its complexity?

Same IDEC-vs-KMeans comparison as the AusNet and London notebooks.

In [ ]:
from sklearn.metrics import davies_bouldin_score, silhouette_score

from ark.cluster.idec import fit_idec

comparison_rows = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=20, random_state=0)
    km_labels = km.fit_predict(X)
    _, idec_labels = fit_idec(X, n_clusters=k, random_state=0)
    comparison_rows.append(
        {
            "k": k,
            "kmeans_silhouette": silhouette_score(X, km_labels),
            "kmeans_davies_bouldin": davies_bouldin_score(X, km_labels),
            "idec_silhouette": silhouette_score(X, idec_labels),
            "idec_davies_bouldin": davies_bouldin_score(X, idec_labels),
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
themed_gt(comparison_df.round(3))

k,kmeans_silhouette,kmeans_davies_bouldin,idec_silhouette,idec_davies_bouldin
2,0.223,1.531,0.26,1.573
3,0.227,1.475,0.203,1.619
4,0.238,1.356,0.148,2.135
5,0.172,1.561,0.166,1.752
6,0.184,1.473,0.131,2.102
7,0.152,1.634,0.128,2.019
8,0.147,1.696,0.122,2.445


In [ ]:
from lets_plot import geom_line, geom_point, scale_color_manual

VALIDITY_METRIC_COLORS = {
    "kmeans_silhouette": BRAND_PALETTE[0],
    "kmeans_davies_bouldin": BRAND_PALETTE[1],
    "idec_silhouette": BRAND_PALETTE[2],
    "idec_davies_bouldin": BRAND_PALETTE[3],
}
scores_normalized = comparison_df.copy()
columns_to_normalize = [c for c in scores_normalized.columns if c != "k"]
scores_normalized[columns_to_normalize] = (
    scores_normalized[columns_to_normalize] - scores_normalized[columns_to_normalize].min()
) / (scores_normalized[columns_to_normalize].max() - scores_normalized[columns_to_normalize].min())

scores_long = scores_normalized.melt(id_vars="k", var_name="metric", value_name="normalized_score")
p = (
    ggplot(scores_long, aes(x="k", y="normalized_score", color="metric"))
    + geom_line()
    + geom_point(size=3)
    + scale_color_manual(values=VALIDITY_METRIC_COLORS)
    + labs(x="Number of clusters (k)", y="Normalized score", color="Score", title="KMeans vs IDEC, by k")
    + ggsize(width=600, height=400)
)
p + modern_theme()

## Are these archetypes stable, or a snapshot artifact?

Same real test as the AusNet and London notebooks: cluster each of the
four real 90-day quarters independently, then check the Adjusted Rand
Index between every pair.

In [ ]:
from sklearn.metrics import adjusted_rand_score

quarters = {"Q1": (0, 90), "Q2": (90, 180), "Q3": (180, 270), "Q4": (270, 360)}
quarter_labels = {}
quarter_silhouettes = {}

for name, (start, end) in quarters.items():
    quarter_x = normalize_shape(load_data[:, start:end, :].mean(axis=1))
    quarter_km = KMeans(n_clusters=N_CLUSTERS, n_init=20, random_state=0)
    quarter_labels[name] = quarter_km.fit_predict(quarter_x)
    quarter_silhouettes[name] = silhouette_score(quarter_x, quarter_labels[name])

quarter_silhouette_df = pd.Series(quarter_silhouettes, name="silhouette").round(3).rename_axis("quarter").reset_index()
themed_gt(quarter_silhouette_df)

quarter,silhouette
Q1,0.235
Q2,0.215
Q3,0.238
Q4,0.186


In [ ]:
quarter_names = list(quarters.keys())
cross_quarter_rows = []
for i in range(len(quarter_names)):
    for j in range(i + 1, len(quarter_names)):
        ari = adjusted_rand_score(quarter_labels[quarter_names[i]], quarter_labels[quarter_names[j]])
        cross_quarter_rows.append({"pair": f"{quarter_names[i]} vs {quarter_names[j]}", "ari": ari})

cross_quarter_df = pd.DataFrame(cross_quarter_rows)
themed_gt(cross_quarter_df.round(3))

pair,ari
Q1 vs Q2,0.335
Q1 vs Q3,0.278
Q1 vs Q4,0.207
Q2 vs Q3,0.465
Q2 vs Q4,0.345
Q3 vs Q4,0.38


In [ ]:
from lets_plot import coord_fixed, geom_text, geom_tile, scale_fill_gradientn

from ark.plot.tokens import BLUES_SEQUENTIAL

transition = pd.crosstab(quarter_labels["Q1"], quarter_labels["Q4"])
transition.index.name = "Q1 archetype"
transition.columns.name = "Q4 archetype"
transition_long = transition.reset_index().melt(id_vars="Q1 archetype", var_name="Q4 archetype", value_name="count")
transition_long["Q1 archetype"] = transition_long["Q1 archetype"].astype(str)
transition_long["Q4 archetype"] = transition_long["Q4 archetype"].astype(str)

p = (
    ggplot(transition_long, aes("Q4 archetype", "Q1 archetype", fill="count"))
    + geom_tile()
    + geom_text(aes(label="count"), color=PRIMARY, size=7)
    + scale_fill_gradientn(colors=BLUES_SEQUENTIAL)
    + coord_fixed(ratio=1)
    + labs(title="Where each Q1 archetype ends up by Q4", x="Q4 archetype", y="Q1 archetype")
    + modern_theme(legend_pos="none")
    + ggsize(420, 420)
)
p

In [ ]:
same_quarter_x = normalize_shape(load_data[:, summer_days, :].mean(axis=1))
seed_labels = [
    KMeans(n_clusters=N_CLUSTERS, n_init=20, random_state=seed).fit_predict(same_quarter_x) for seed in range(5)
]

seed_ari_rows = []
for i in range(len(seed_labels)):
    for j in range(i + 1, len(seed_labels)):
        ari = adjusted_rand_score(seed_labels[i], seed_labels[j])
        seed_ari_rows.append({"pair": f"seed{i} vs seed{j}", "ari": ari})

seed_ari_df = pd.DataFrame(seed_ari_rows)
themed_gt(seed_ari_df.round(3))

pair,ari
seed0 vs seed1,1.0
seed0 vs seed2,1.0
seed0 vs seed3,0.979
seed0 vs seed4,1.0
seed1 vs seed2,1.0
seed1 vs seed3,0.979
seed1 vs seed4,1.0
seed2 vs seed3,0.979
seed2 vs seed4,1.0
seed3 vs seed4,0.979


In [ ]:
from lets_plot import geom_jitter

COMPARISON_COLORS = {"Different quarters": BRAND_PALETTE[0], "Same quarter, different seed": BRAND_PALETTE[1]}

ari_comparison = pd.concat(
    [
        cross_quarter_df.assign(comparison="Different quarters"),
        seed_ari_df.assign(comparison="Same quarter, different seed"),
    ],
    ignore_index=True,
)

p = (
    ggplot(ari_comparison, aes(x="comparison", y="ari", color="comparison"))
    + geom_jitter(width=0.15, height=0, size=4)
    + scale_color_manual(values=COMPARISON_COLORS)
    + labs(
        x="",
        y="Adjusted Rand Index",
        title="Real drift, not clustering noise",
        subtitle="Each point is one pair of independent clusterings",
    )
    + modern_theme(legend_pos="none")
    + ggsize(500, 400)
)
p

## Sensitivity: how much history does one clustering run need?

Same real sweep as the AusNet and London notebooks: 1 day, 1 week, 1
month, 1 season, each re-clustered from scratch and checked against the
90-day season baseline above.

In [ ]:
windows = {
    "1 day": load_data[:, 0:1, :],
    "1 week": load_data[:, 0:7, :],
    "1 month (30 days)": load_data[:, 0:30, :],
    "1 season (90 days)": season,
}

window_ari_vs_season = {}
window_silhouette = {}
for name, window in windows.items():
    window_x = normalize_shape(window.mean(axis=1))
    window_km = KMeans(n_clusters=N_CLUSTERS, n_init=20, random_state=0)
    window_labels = window_km.fit_predict(window_x)
    window_ari_vs_season[name] = adjusted_rand_score(labels, window_labels)
    window_silhouette[name] = silhouette_score(window_x, window_labels)

window_df = pd.DataFrame({"ARI vs season baseline": window_ari_vs_season, "Silhouette": window_silhouette})
themed_gt(window_df.round(3).rename_axis("window").reset_index())

window,ARI vs season baseline,Silhouette
1 day,0.142,0.22
1 week,0.246,0.188
1 month (30 days),0.491,0.23
1 season (90 days),1.0,0.238


In [ ]:
from lets_plot import facet_wrap

window_order = ["1 day", "1 week", "1 month (30 days)", "1 season (90 days)"]
window_long = (
    window_df.reset_index()
    .rename(columns={"index": "window"})
    .melt(id_vars="window", var_name="metric", value_name="value")
)
window_long["window"] = pd.Categorical(window_long["window"], categories=window_order, ordered=True)

p = (
    ggplot(window_long, aes(x="window", y="value"))
    + geom_line(aes(group="metric"), color=BRAND_PALETTE[0])
    + geom_point(color=BRAND_PALETTE[0], size=4)
    + facet_wrap("metric", ncol=2, scales="free_y")
    + labs(x="History used for one clustering run", y="", title="More history, more trustworthy archetypes")
    + modern_theme(x_axis_angle=25, legend_pos="none")
    + ggsize(700, 400)
)
p

## Summary

GoiEner's own real run does not simply slot in between AusNet and London;
it disagrees with both on the one point they agreed on.

AusNet and London's own real validity curves both fall monotonically as
$k$ grows, no clean elbow either way. GoiEner's does not: silhouette rises
from 0.223 ($k{=}2$) through 0.227 ($k{=}3$) to a real peak of 0.238 at
$k{=}4$, then drops sharply to 0.172 at $k{=}5$, a genuine non-monotonic
shape neither other population produces. The chosen $k{=}4$ splits 2,000
households into real groups of 1,153, 546, 235, and 66, still a lopsided
split proportionally (the smallest group is 3.3% of the population,
similar to the 2.3% found at the earlier 300-household scale) but now
anchored on a real, statistically credible minority of 66 households
rather than a fragile 7.

Cross-quarter ARI (0.207-0.465) now sits genuinely between AusNet's own
low-stability finding (0.06-0.34) and London's meaningfully higher one
(0.417-0.602), touching the upper edge of one and the lower edge of the
other rather than sitting closer to either: GoiEner's real archetypes
drift quarter to quarter somewhere between AusNet's and London's own
real rates, not distinctly toward one. Same-quarter, different-seed ARI
(0.979-1.000) is real and high, confirming that drift is genuine
population change rather than clustering noise, and at this larger scale
it is now just as clean as the near-perfect 0.92-1.0 both AusNet and
London achieve: the larger, statistically credible minority cluster gives
KMeans far less room to disagree with itself across random seeds than the
earlier 7-member cluster did. History-length sensitivity still points the
same direction as both other populations (more history buys more stable
archetypes): 1 day (0.142), 1 week (0.246), 1 month (0.491), against
AusNet and London's own comparable numbers.

IDEC only looks competitive with KMeans at $k{=}2$ (0.260 vs 0.223,
IDEC actually ahead); at the real chosen $k{=}4$, KMeans wins clearly
(0.238 vs 0.148), the same real conclusion AusNet and London both reach
once $k$ is chosen properly rather than read off a single favorable row.

GoiEner's real hourly (not half-hourly) resolution and its Basque
Country/Navarre-heavy geography (not broad Spain-wide climate diversity)
are both real, disclosed differences from AusNet and London; whether they
explain the non-monotonic validity curve and the lopsided cluster sizes,
or a different real population mechanism does, is an open question this
2,000-household run does not resolve either, though the larger sample
does rule out one candidate explanation: the non-monotonic curve and the
lopsided split are not artifacts of a small n=300 sample, since both
persist at nearly 7x the population.